In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY não encontrada no arquivo .env")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

In [ ]:
df = pd.read_csv("../data/raw/dataset.csv")
df.head()

In [ ]:
texto = df["Text"].iloc[0]
print(texto)

In [ ]:
prompt = f"""
Você é um especialista em avaliação educacional.

Classifique o feedback nas categorias abaixo. Para cada uma, responda 1 (presente) ou 0 (ausente).

GP1: Esclarece o que é um bom desempenho.
GP2: Promove autoavaliação e reflexão do aluno.
GP3: Fornece informações de qualidade sobre o aprendizado.
GP4: Incentiva diálogo entre aluno e professor/colegas.
GP5: Incentiva motivação e autoestima.
GP6: Ajuda a reduzir a diferença entre desempenho atual e desejado.
GP7: Fornece informações úteis para o professor ajustar o ensino.

Responda no formato:
GP1: 0 ou 1
GP2: 0 ou 1
GP3: 0 ou 1
GP4: 0 ou 1
GP5: 0 ou 1
GP6: 0 ou 1
GP7: 0 ou 1

Feedback:
{texto}
"""

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
)
print(response.choices[0].message.content)

## Processamento em lote (Batch API)

A OpenAI também tem uma [Batch API](https://platform.openai.com/docs/guides/batch), que processa várias requisições de uma vez de forma assíncrona.

Vantagens:
- **50% mais barato** que chamadas individuais.
- Até 50.000 requisições por lote.
- Ótimo para o nosso caso: 1000 feedbacks no dataset — nada de ficar chamando a API um por um.

O fluxo é um pouco diferente do Claude/Gemini: em vez de mandar as requisições direto, a OpenAI pede um arquivo no formato JSONL (uma requisição por linha, cada uma com um `custom_id`). O fluxo é: (1) montar o JSONL e enviar via Files API; (2) criar o lote apontando para o arquivo; (3) aguardar o processamento; (4) baixar o arquivo de resultados e casar cada resposta com seu `custom_id` (a ordem das respostas não é garantida).

**Abaixo, o lote roda com o dataset completo (1000 feedbacks).** O tempo de processamento não é garantido — pode levar de poucos minutos até (no pior caso) 24h. Se quiser testar mais rápido antes de rodar tudo, troque `linhas = df` por `linhas = df.head(5)`.

In [ ]:
def montar_prompt(texto: str) -> str:
    return f"""
Você é um especialista em avaliação educacional.

Classifique o feedback nas categorias abaixo. Para cada uma, responda 1 (presente) ou 0 (ausente).

GP1: Esclarece o que é um bom desempenho.
GP2: Promove autoavaliação e reflexão do aluno.
GP3: Fornece informações de qualidade sobre o aprendizado.
GP4: Incentiva diálogo entre aluno e professor/colegas.
GP5: Incentiva motivação e autoestima.
GP6: Ajuda a reduzir a diferença entre desempenho atual e desejado.
GP7: Fornece informações úteis para o professor ajustar o ensino.

Responda no formato:
GP1: 0 ou 1
GP2: 0 ou 1
GP3: 0 ou 1
GP4: 0 ou 1
GP5: 0 ou 1
GP6: 0 ou 1
GP7: 0 ou 1

Feedback:
{texto}
"""

In [ ]:
import io
import json
import time

linhas = df  # dataset completo (1000 feedbacks). Para testar rápido, use: df.head(5)

# Monta o arquivo JSONL em memória (uma requisição por linha)
buffer = io.BytesIO()
for row in linhas.itertuples():
    linha = {
        "custom_id": f"feedback-{row.ID}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": MODEL,
            "messages": [{"role": "user", "content": montar_prompt(row.Text)}],
        },
    }
    buffer.write((json.dumps(linha) + "\n").encode("utf-8"))
buffer.seek(0)
buffer.name = "batch_requests.jsonl"

arquivo_lote = client.files.create(file=buffer, purpose="batch")
print(f"Arquivo enviado: {arquivo_lote.id}")

In [ ]:
batch = client.batches.create(
    input_file_id=arquivo_lote.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)
print(f"Lote criado: {batch.id} (status: {batch.status})")

In [ ]:
estados_finais = {"completed", "failed", "expired", "cancelled"}

while batch.status not in estados_finais:
    print(f"Status atual: {batch.status} ({batch.request_counts})")
    time.sleep(15)
    batch = client.batches.retrieve(batch.id)

print(f"Lote concluído com status: {batch.status}")
print(batch.request_counts)

In [ ]:
# As respostas não vêm necessariamente na mesma ordem das requisições —
# por isso casamos cada uma pelo custom_id
resultados = {}

if batch.status == "completed" and batch.output_file_id:
    conteudo = client.files.content(batch.output_file_id).text
    for linha in conteudo.splitlines():
        item = json.loads(linha)
        custom_id = item["custom_id"]
        if item.get("error"):
            resultados[custom_id] = f"[erro: {item['error']}]"
        else:
            resultados[custom_id] = item["response"]["body"]["choices"][0]["message"]["content"]

    for custom_id in sorted(resultados):
        print(f"--- {custom_id} ---")
        print(resultados[custom_id])
        print()
else:
    print(f"Lote não concluído com sucesso: status={batch.status}, erros={batch.errors}")